In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================
# CONFIGURATION
# =========================================================
series_list = 'series/'
test_h5_path = 'hrv_test.h5'  # Adjust path if necessary
model_path = 'cnn_lstm_hrv_generalist.keras'

files_test = [
    "4016.txt", "4005.txt", "4062.txt", "4099.txt", "4012.txt",
    "4085.txt", "4086.txt", "403.txt", "16786.txt",
    "nsr010RRcl.txt", "nsr003RRcl.txt"
]

percents_array = np.arange(0.01, 0.85, 0.05)

feature_cols = ["mean", "sdsd", "sd2", "ccm", "guzik", "nn50", "porta", "std"]
rr_cols = [f"rr_{i}" for i in range(1, 21)]


# =========================================================
# UTILS & MODIFIED EVALUATION FUNCTION
# =========================================================
def canonical_code(x):
    """Normalize subject codes to match H5 index definitions."""
    s = str(x).strip()
    if s.isdigit():
        return f"{int(s):03d}" if len(s) < 3 else s
    return s

@tf.function(reduce_retracing=True)
def fast_predict(seq_input, feats_input, loaded_model):
    return loaded_model([seq_input, feats_input], training=False)

def random_extraction(original_serie, percent_to_eliminate, start_idx=40):
    n_total = len(original_serie)
    if start_idx >= n_total:
        raise ValueError(f"start_idx ({start_idx}) cannot be >= series length ({n_total}).")
        
    n_to_eliminate = int(round(n_total * percent_to_eliminate))
    available_spots = n_total - start_idx
    
    if n_to_eliminate > available_spots:
        raise ValueError(f"Cannot eliminate {n_to_eliminate} values. Only {available_spots} valid spots available.")

    modified_serie = np.array(original_serie, dtype=float)
    random_indexes = np.random.choice(np.arange(start_idx, n_total), size=n_to_eliminate, replace=False)
    modified_serie[random_indexes] = np.nan

    return modified_serie

def extract_hrv_features(serie, window_size=20, window_size_long=40):
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)
    ventanas_long = sliding_window_view(serie, window_size_long)
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]
    X_ventanas_short = X_ventanas_long[:, -window_size:]
    diffs = np.diff(X_ventanas_short, axis=1)

    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    nn50 = np.sum(np.abs(diffs) > 50, axis=1)
    nn20 = np.sum(np.abs(diffs) > 20, axis=1)

    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2)

    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    std_long = np.std(X_ventanas_long, axis=1)
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))
    c_n = np.pi * sd1 * sd2

    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i, rr_i1, rr_i2, rr_i3 = ventanas_4puntos[:, 0], ventanas_4puntos[:, 1], ventanas_4puntos[:, 2], ventanas_4puntos[:, 3]
    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))
    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    cv = np.divide(std_val, mean_val, out=np.zeros_like(std_val, dtype=float), where=mean_val!=0)

    q75, q25 = np.percentile(X_ventanas_short, [75, 25], axis=1)
    iqr = q75 - q25

    median_val = np.median(X_ventanas_short, axis=1)
    mad = np.median(np.abs(X_ventanas_short - median_val[:, None]), axis=1)

    diffs_1 = diffs[:, :-1]
    diffs_2 = diffs[:, 1:]
    inflections = (diffs_1 * diffs_2) <= 0
    pip = np.sum(inflections, axis=1) / (window_size - 2)

    mean_diffs = np.mean(diffs, axis=1, keepdims=True)
    std_diffs = np.std(diffs, axis=1, keepdims=True)
    std_diffs_safe = np.where(std_diffs == 0, 1e-10, std_diffs)
    skewness = np.mean(((diffs - mean_diffs) / std_diffs_safe)**3, axis=1)

    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}
    stats_columns = {
        'n_above': n_above, 'n_below': n_below, 'nn20': nn20, 'nn50': nn50,
        'sdsd': sdsd, 'mean': mean_val, 'std': std_val, 'var': var_val,
        'std_long': std_long, 'sd1': sd1, 'sd2': sd2, 'c_n': c_n,
        'ccm': ccm, 'porta': porta_index, 'guzik': guzic_index,
        'cv': cv, 'iqr': iqr, 'mad': mad, 'pip': pip, 'skewness': skewness,
        'target': y_target
    }

    return pd.DataFrame({**rr_columns, **stats_columns})


def evaluate_imputation_performance(original_serie, percents_to_eliminate, loaded_model, feats_mean, feats_scale, seq_mean, seq_scale, y_mean, y_scale, feature_cols, rr_cols):
    """
    Evaluates autoregressive imputation performance of a model.
    Modified to explicitly return MSE alongside other metrics.
    """
    print(f"Starting autoregressive imputation across {len(percents_to_eliminate)} thresholds...")
    
    rmse = np.zeros_like(percents_to_eliminate, dtype=float)
    mse = np.zeros_like(percents_to_eliminate, dtype=float)  # Newly Added
    mae = np.zeros_like(percents_to_eliminate, dtype=float)
    r2 = np.zeros_like(percents_to_eliminate, dtype=float)
    correlation = np.zeros_like(percents_to_eliminate, dtype=float)

    for idx, percent in enumerate(percents_to_eliminate):
        
        y_true = []  
        y_pred = []
        
        modified_serie = random_extraction(original_serie, percent_to_eliminate=percent, start_idx=40)
        
        for i in range(40, len(modified_serie)):
            if np.isnan(modified_serie[i]):
                
                history_clean = modified_serie[i-40 : i]
                window_data_for_func = np.append(history_clean, np.nan)
                
                df_step = extract_hrv_features(window_data_for_func, window_size=20, window_size_long=40)
                
                X_feats_step = df_step[feature_cols].values
                X_rr_seq_step = df_step[rr_cols].values
                
                X_feats_step_scaled = (X_feats_step - feats_mean) / feats_scale
                X_rr_seq_step_scaled = (X_rr_seq_step - seq_mean) / seq_scale
                
                X_rr_seq_step_3d = X_rr_seq_step_scaled.reshape(1, 20, 1)
                
                y_pred_diff_scaled_tensor = fast_predict(
                    tf.convert_to_tensor(X_rr_seq_step_3d, dtype=tf.float32), 
                    tf.convert_to_tensor(X_feats_step_scaled, dtype=tf.float32),
                    loaded_model
                )
                y_pred_diff_scaled = y_pred_diff_scaled_tensor.numpy()

                y_pred_diff_ms = (y_pred_diff_scaled.flatten()[0] * y_scale) + y_mean
                y_pred_ms = y_pred_diff_ms + history_clean[-1] 
                
                y_true.append(original_serie[i])  
                y_pred.append(y_pred_ms)  
                modified_serie[i] = y_pred_ms

        if len(y_true) > 0:
            rmse[idx] = np.sqrt(mean_squared_error(y_true, y_pred))
            mse[idx] = mean_squared_error(y_true, y_pred) # Newly added
            mae[idx] = mean_absolute_error(y_true, y_pred)
            r2[idx] = r2_score(y_true, y_pred)
        else:
            rmse[idx], mse[idx], mae[idx], r2[idx] = np.nan, np.nan, np.nan, np.nan
            
        correlation[idx] = np.corrcoef(original_serie, modified_serie)[0, 1]
                
    return rmse, mse, mae, r2, correlation


# =========================================================
# MAIN EXECUTION
# =========================================================
print("Loading model...")
loaded_model = tf.keras.models.load_model(model_path)

print(f"Opening HDF5 dataset to extract normalization logic: {test_h5_path}")
with h5py.File(test_h5_path, "r") as h5:
    
    # Pre-map index for fast lookups
    idx_subjects = [s.decode('utf-8') if isinstance(s, bytes) else str(s) for s in h5["index"]["subject_id"][()]]
    idx_paths = [p.decode('utf-8') if isinstance(p, bytes) else str(p) for p in h5["index"]["h5_path"][()]]
    subject_to_path = dict(zip(idx_subjects, idx_paths))
    
    for file in files_test:
        subject_name = os.path.splitext(file)[0]
        s_code = canonical_code(subject_name)
        
        if s_code not in subject_to_path:
            print(f"⚠️ Subject {s_code} ({file}) not found in test index. Skipping.")
            continue
            
        print(f"\n--- Processing Subject: {subject_name} ---")
        
        # Extract normalization references mapped during split
        h5_path = subject_to_path[s_code]
        subj_group = h5[h5_path]
        
        rr_mean = float(subj_group.attrs["rr_mean_ref"])
        rr_scale = float(subj_group.attrs["rr_std_ref"])
        
        interval = subj_group.attrs["interval"]
        interval = interval.decode('utf-8') if isinstance(interval, bytes) else str(interval)
            
        norm_group = h5[f"normalization/{interval}"]
        norm_cols = [c.decode('utf-8') if isinstance(c, bytes) else str(c) for c in norm_group["columns"][()]]
        norm_mean = norm_group["mean"][()]
        norm_std = norm_group["std"][()]
        
        y_mean = float(norm_group.attrs["target_mean"])
        y_scale = float(norm_group.attrs["target_std"])
        
        # Build feature arrays mirroring the precise model input definition
        feats_mean = np.zeros(len(feature_cols))
        feats_scale = np.zeros(len(feature_cols))
        
        for i, col in enumerate(feature_cols):
            if col == "mean":
                feats_mean[i] = rr_mean
                feats_scale[i] = rr_scale
            else:
                col_idx = norm_cols.index(col)
                feats_mean[i] = norm_mean[col_idx]
                feats_scale[i] = norm_std[col_idx]
                
        # Run imputation evaluation
        file_path = os.path.join(series_list, file)
        original_serie = np.loadtxt(file_path, dtype=int).astype(float)
        
        rmse_res, mse_res, mae_res, r2_res, corr_res = evaluate_imputation_performance(
            original_serie=original_serie,
            percents_to_eliminate=percents_array,
            loaded_model=loaded_model,
            feats_mean=feats_mean, feats_scale=feats_scale,
            seq_mean=rr_mean, seq_scale=rr_scale,
            y_mean=y_mean, y_scale=y_scale,
            feature_cols=feature_cols, rr_cols=rr_cols
        )
        
        # Package and export
        results_df = pd.DataFrame({
            'Percent_Eliminated': percents_array,
            'RMSE': rmse_res,
            'MSE': mse_res,
            'MAE': mae_res,
            'R2': r2_res,
            'Correlation': corr_res
        })
        
        out_csv = f'imputation_metrics_{subject_name}.csv'
        results_df.to_csv(out_csv, index=False)
        print(f"Results saved to: {out_csv}")

I0000 00:00:1785525917.889174  960227 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785525923.015702  960227 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loading model...


ValueError: File not found: filepath=cnn_lstm_hrv_best.keras. Please ensure the file is an accessible `.keras` zip file.